# Практика · YOLO: детекція за один прохід

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі дані зошит генерує формулами. Ваги YOLO не
> завантажуються — ми будуємо **власну маленьку YOLO** й розбираємо її по числах.
> Досить `torch`, `torchvision`, `numpy` і `matplotlib`.

> ⏱ Зошит навчає **шість** мереж (дві настройки втрати × три зерна). Заміряно:
> **близько двох з половиною хвилин** на чотирьох ядрах без відеокарти. Найдовше
> йде розділ 8 — приблизно 19 секунд на кожне навчання.

Що зробимо:

1. порахуємо **розмір виходу** `S×S×(B·5+C)` для різних сіток;
2. напишемо **своє кодування** цілі: клітинка відповідає за предмет, чий центр у ній;
3. напишемо **своє декодування** виходу в рамки й звіримо його руками та кільцевою перевіркою;
4. поміряємо, **скільки предметів сітка втрачає** через конфлікт центрів при `S` = 4, 7, 8, 13;
5. побачимо в числах, **навіщо корінь** із ширини й висоти;
6. розкладемо втрату YOLO на складники й перевіримо, **навіщо `λ_coord` і `λ_noobj`**;
7. навчимо шість мереж і порівняємо `mAP@0.5` двох настройок утрати;
8. заміряємо **швидкість** одноетапного проходу проти двоетапної схеми;
9. зберемо офлайн голову YOLOv8 і подивимось, що в ній лишилось від першої версії.

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
from torchvision.ops import box_iou, nms
import matplotlib.pyplot as plt

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)
torch.manual_seed(0)

print("torch     ", torch.__version__)
print("numpy     ", np.__version__)
print("потоків   ", torch.get_num_threads())

## 1 · Датасет: ті самі сцени 64×64

Беремо той самий наскрізний приклад, що й у [темі 23](../23-detection-setup/lecture.html):
полотно 64 на 64 пікселі, від одного до трьох предметів трьох класів (коло, квадрат,
трикутник) радіусом 6-10 пікселів, шум зі стандартним відхиленням 0.12. Рамка кожного
предмета рахується **з його маски**, тому вона істинна за побудовою.

Окрім звичайного набору згенеруємо ще й **щільний**: чотири-шість дрібніших предметів
на тому самому полотні. На ньому ми поміряємо головне обмеження сітки.

In [ ]:
SIZE = 64                                  # сторона полотна в пікселях
CLASS_NAMES = ["коло", "квадрат", "трикутник"]
CLASS_COUNT = 3


def shape_mask(kind, center_x, center_y, radius):
    """Маска однієї фігури на полотні 64×64."""
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                   # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                   # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def make_scene(rng, min_objects=1, max_objects=4, min_radius=6, max_radius=11):
    """Одна сцена: картинка, рамки з масок, мітки класів."""
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels = [], []
    for _ in range(int(rng.integers(min_objects, max_objects))):
        for _attempt in range(40):
            radius = int(rng.integers(min_radius, max_radius))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            mask = shape_mask(kind, center_x, center_y, radius)
            ys, xs = np.nonzero(mask)
            # рамка береться з маски: край + 1, як в угоді COCO
            box = [float(xs.min()), float(ys.min()),
                   float(xs.max() + 1), float(ys.max() + 1)]

            # не даємо предметам злипатись більше ніж на 45 % площі нового
            too_close = False
            for previous in boxes:
                overlap_w = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                overlap_h = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if overlap_w * overlap_h > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue

            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return image, np.array(boxes, np.float32), np.array(labels, np.int64)


def make_dataset(seed, count, **scene_options):
    rng = np.random.default_rng(seed)
    return [make_scene(rng, **scene_options) for _ in range(count)]


started = time.time()
train_set = make_dataset(42, 400)
test_set = make_dataset(7, 120)
# великий набір саме для статистики конфліктів — навчати на ньому не будемо
sparse_set = make_dataset(123, 1000)
dense_set = make_dataset(5, 1000, min_objects=4, max_objects=7,
                         min_radius=4, max_radius=8)

true_box_count = sum(len(scene[1]) for scene in test_set)
print("згенеровано за %.2f с" % (time.time() - started))
print("навчальних сцен %d, перевірних %d" % (len(train_set), len(test_set)))
print("істинних рамок у перевірному наборі: %d" % true_box_count)
print()
print("рідкі сцени:  %d предметів на 1000 сцен (%.2f на сцену)"
      % (sum(len(s[1]) for s in sparse_set),
         sum(len(s[1]) for s in sparse_set) / 1000))
print("щільні сцени: %d предметів на 1000 сцен (%.2f на сцену)"
      % (sum(len(s[1]) for s in dense_set),
         sum(len(s[1]) for s in dense_set) / 1000))

## 2 · Розмір виходу: `S × S × (B·5 + C)`

Уся відповідь YOLO — **один тензор фіксованого розміру**. Сітка `S` на `S` клітинок,
на кожну клітинку `B` рамок по пʼять чисел (`x`, `y`, `w`, `h`, впевненість) плюс `C`
чисел класів на всю клітинку.

Порахуємо цей розмір руками для кількох конфігурацій — зокрема для справжньої
YOLOv1 на наборі Pascal VOC (`S=7`, `B=2`, `C=20`).

In [ ]:
print("  S   B   C     форма         чисел    що це")
print("  " + "-" * 58)
for grid_size, boxes_per_cell, class_count, note in (
        (7, 2, 20, "YOLOv1 на Pascal VOC"),
        (8, 2, 3, "наша мережа"),
        (4, 2, 3, ""),
        (13, 2, 3, ""),
        (8, 1, 3, "одна рамка на клітинку"),
        (8, 3, 3, "три рамки на клітинку"),
        (13, 5, 20, "голова у стилі YOLOv2")):
    depth = boxes_per_cell * 5 + class_count
    total = grid_size * grid_size * depth
    print(" %2d   %d  %2d   %2d×%2d×%2d   %8d    %s"
          % (grid_size, boxes_per_cell, class_count,
             grid_size, grid_size, depth, total, note))

print()
print("Глибина не залежить від S: додаємо клітинок — росте лише перших два виміри.")
print("Додаємо рамку на клітинку — глибина росте на 5, і то для всієї сітки одразу.")

## 3 · Кодування цілі: хто за що відповідає

Правило YOLO коротке: **клітинка відповідає за предмет, чий центр у неї потрапив**.
Не «перетинається», не «лежить усередині» — саме центр.

Що записуємо в ціль для відповідальної клітинки:

- `x`, `y` — де центр предмета **всередині клітинки**, числа від 0 до 1;
- `√w`, `√h` — корінь із ширини й висоти, поділених на розмір зображення;
- клас предмета.

Якщо клітинка вже зайнята іншим предметом, новий предмет **просто зникає з цілі**.
Ми ще й порахуємо, скільки разів це стається.

In [ ]:
GRID = 8                                   # S
BOXES = 2                                  # B
CELL = SIZE / GRID                         # сторона клітинки в пікселях


def encode_targets(boxes, labels, grid_size=GRID):
    """Ціль YOLO: не більше одного предмета на клітинку.

    Повертає три масиви й кількість предметів, які клітинка не змогла взяти.
    """
    cell = SIZE / grid_size
    objectness = np.zeros((grid_size, grid_size), np.float32)
    target = np.zeros((grid_size, grid_size, 4), np.float32)
    class_target = np.zeros((grid_size, grid_size), np.int64)
    dropped = 0

    for (x1, y1, x2, y2), label in zip(boxes, labels):
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2
        col = int(center_x // cell)
        row = int(center_y // cell)

        if objectness[row, col] > 0:
            # клітинка вже відповідає за інший предмет — цей втрачено назавжди
            dropped += 1
            continue

        objectness[row, col] = 1.0
        # зсув центра ВСЕРЕДИНІ клітинки: 0 — лівий край клітинки, 1 — правий
        target[row, col, 0] = center_x / cell - col
        target[row, col, 1] = center_y / cell - row
        # розмір — частка від усього зображення, і одразу під коренем
        target[row, col, 2] = math.sqrt((x2 - x1) / SIZE)
        target[row, col, 3] = math.sqrt((y2 - y1) / SIZE)
        class_target[row, col] = label

    return objectness, target, class_target, dropped


# подивимось на одну сцену очима: яка клітинка за що відповідає
image, boxes, labels = test_set[0]
objectness, target, class_target, dropped = encode_targets(boxes, labels)

print("предметів у сцені:", len(boxes))
for (x1, y1, x2, y2), label in zip(boxes, labels):
    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2
    col = int(center_x // CELL)
    row = int(center_y // CELL)
    print("  %-10s центр (%5.1f, %5.1f) → клітинка (рядок %d, стовпець %d), "
          "зсув усередині (%.3f, %.3f)"
          % (CLASS_NAMES[label], center_x, center_y, row, col,
             center_x / CELL - col, center_y / CELL - row))

print()
print("позитивних клітинок у цілі: %d із %d" % (int(objectness.sum()), GRID * GRID))
print("предметів, які не влізли:   %d" % dropped)

## 4 · Декодування: із сітки назад у рамки

Це найважливіша функція теми. Мережа видає числа в системі координат клітинки —
щоб отримати рамку в пікселях, треба зробити чотири кроки:

1. `center_x = (col + x) · cell` — до номера клітинки додати зсув усередині неї;
2. те саме для `center_y`;
3. `w = (√w)² · SIZE` — підняти корінь у квадрат і помножити на розмір зображення;
4. з центра й розміру зібрати `xyxy`.

Спершу порахуємо один приклад **руками**, потім напишемо функцію й перевіримо, що
вона дає те саме число в число.

In [ ]:
# приклад руками: клітинка (рядок 3, стовпець 5), зсув (0.25, 0.75),
# корінь ширини 0.50, корінь висоти 0.60
example_row, example_col = 3, 5
example = [0.25, 0.75, 0.50, 0.60]

hand_center_x = (example_col + example[0]) * CELL
hand_center_y = (example_row + example[1]) * CELL
hand_width = (example[2] ** 2) * SIZE
hand_height = (example[3] ** 2) * SIZE
hand_box = [hand_center_x - hand_width / 2, hand_center_y - hand_height / 2,
            hand_center_x + hand_width / 2, hand_center_y + hand_height / 2]

print("клітинка (%d, %d), сторона клітинки %.0f px" % (example_row, example_col, CELL))
print("центр   x = (%d + %.2f) × %.0f = %.1f" % (example_col, example[0], CELL, hand_center_x))
print("центр   y = (%d + %.2f) × %.0f = %.1f" % (example_row, example[1], CELL, hand_center_y))
print("ширина    = %.2f² × %d = %.2f" % (example[2], SIZE, hand_width))
print("висота    = %.2f² × %d = %.2f" % (example[3], SIZE, hand_height))
print("рамка xyxy: [%.2f, %.2f, %.2f, %.2f]" % tuple(hand_box))

Тепер та сама арифметика у вигляді функції, яка обробляє всю сітку одразу.
`coords` має форму `(N, S, S, B, 4)` — по чотири числа на кожну з `B` рамок кожної
клітинки кожної сцени.

In [ ]:
def decode_boxes(coords, grid_size=GRID):
    """(N, S, S, B, 4) з числами 0…1 → рамки xyxy у пікселях."""
    cell = SIZE / grid_size
    # номери стовпців і рядків розкладаємо по потрібних вимірах, щоб
    # додавання зсуву працювало для всієї сітки одним виразом
    cols = torch.arange(grid_size, dtype=torch.float32).view(1, 1, grid_size, 1)
    rows = torch.arange(grid_size, dtype=torch.float32).view(1, grid_size, 1, 1)

    center_x = (cols + coords[..., 0]) * cell
    center_y = (rows + coords[..., 1]) * cell
    width = (coords[..., 2] ** 2) * SIZE          # корінь підіймаємо назад у квадрат
    height = (coords[..., 3] ** 2) * SIZE

    return torch.stack([center_x - width / 2, center_y - height / 2,
                        center_x + width / 2, center_y + height / 2], dim=-1)


# кладемо наш приклад у порожню сітку й дивимось, чи функція згодна з рукою
probe = torch.zeros(1, GRID, GRID, BOXES, 4)
probe[0, example_row, example_col, 0] = torch.tensor(example)
decoded = decode_boxes(probe)[0, example_row, example_col, 0].tolist()

print("руками  ", ["%.2f" % v for v in hand_box])
print("функцією", ["%.2f" % v for v in decoded])
assert np.allclose(decoded, hand_box, atol=1e-4), "декодування розійшлось із ручним підрахунком!"
print("✅ збігається")

Друга перевірка, сильніша за першу: закодуємо **всі істинні рамки** перевірного набору
й одразу розкодуємо назад. Якщо кодування й декодування — це справді одне й те саме
перетворення в два боки, ми маємо отримати вихідні рамки з точністю до округлення
`float32`.

In [ ]:
biggest_error = 0.0
checked = 0

for image, boxes, labels in test_set:
    objectness, target, class_target, dropped = encode_targets(boxes, labels)
    # кладемо ціль у форму, яку чекає decode_boxes: одна рамка на клітинку
    grid_input = torch.from_numpy(target).unsqueeze(0).unsqueeze(3)
    restored = decode_boxes(grid_input)[0, :, :, 0]

    for (x1, y1, x2, y2) in boxes:
        col = int(((x1 + x2) / 2) // CELL)
        row = int(((y1 + y2) / 2) // CELL)
        if objectness[row, col] == 0:
            continue
        got = restored[row, col].tolist()
        error = max(abs(got[0] - x1), abs(got[1] - y1), abs(got[2] - x2), abs(got[3] - y2))
        biggest_error = max(biggest_error, error)
        checked += 1

print("перевірено рамок:        %d" % checked)
print("найбільше розходження:   %.2e пікселя" % biggest_error)
assert biggest_error < 1e-3, "кодування й декодування не є оберненими одне до одного!"
print("✅ кодування та декодування обертають одне одного")

## 5 · Головний замір теми: скільки предметів губить сітка

Правило «одна клітинка — один предмет» має ціну. Якщо центри двох предметів впали
в одну клітинку, **другий предмет для мережі не існує**: його немає в цілі, тож
мережа ніколи не дістане за нього градієнта й ніколи не навчиться його бачити.
Це не помилка навчання й не поганий поріг — це вроджене обмеження постановки.

Порахуємо втрату на двох наборах по 1000 сцен: рідкому (1-3 предмети) і щільному
(4-6 дрібніших). Сітки беремо `S` = 4, 7, 8 і 13 — сімка тут не випадкова, саме така
сітка була в першій YOLO.

In [ ]:
def count_conflicts(dataset, grid_size):
    """Скільки предметів втрачається, бо їхній центр упав у зайняту клітинку."""
    cell = SIZE / grid_size
    objects_total = 0
    objects_lost = 0
    scenes_damaged = 0

    for _image, boxes, _labels in dataset:
        taken = set()
        lost_here = 0
        for (x1, y1, x2, y2) in boxes:
            objects_total += 1
            key = (int(((y1 + y2) / 2) // cell), int(((x1 + x2) / 2) // cell))
            if key in taken:
                lost_here += 1
            else:
                taken.add(key)
        objects_lost += lost_here
        if lost_here:
            scenes_damaged += 1

    return objects_total, objects_lost, scenes_damaged


for name, dataset in (("рідкі (1-3 предмети)", sparse_set),
                      ("щільні (4-6 предметів)", dense_set)):
    print(name)
    print("   S   клітинка   предметів   втрачено      %    зіпсовано сцен")
    for grid_size in (4, 7, 8, 13):
        total, lost, damaged = count_conflicts(dataset, grid_size)
        print("  %2d   %5.1f px      %6d       %5d   %5.2f    %4d (%4.1f %%)"
              % (grid_size, SIZE / grid_size, total, lost,
                 100 * lost / total, damaged, 100 * damaged / len(dataset)))
    print()

Числа малі — і це чесно: наші сцени рідкі, у них щонайбільше три предмети, та ще й
генератор не дає їм сильно злипатись. Справжня вулична фотографія так себе не поводить.

Порахуємо, чого чекати на щільній сцені, **не генеруючи її**. Задача та сама, що й
у відомому «парадоксі днів народження»: якщо `k` предметів кидати в `S²` клітинок
навмання, імовірність, що хоча б двоє попадуть в одну, дорівнює

`1 − (n/n) · ((n−1)/n) · … · ((n−k+1)/n)`, де `n = S²`.

Це верхня оцінка: справжні предмети розкидані не зовсім рівномірно. Але вона показує,
куди все йде.

In [ ]:
print("предметів   S=4 (16 кл.)   S=7 (49)   S=8 (64)   S=13 (169)")
for object_count in (2, 3, 5, 10, 20, 30):
    row = []
    for grid_size in (4, 7, 8, 13):
        cells = grid_size * grid_size
        no_clash = 1.0
        for i in range(object_count):
            no_clash *= (cells - i) / cells
        row.append(100 * (1 - no_clash))
    print("   %2d       %8.1f %%   %7.1f %%   %6.1f %%   %8.1f %%"
          % (object_count, row[0], row[1], row[2], row[3]))

print()
print("На фотографії з двома десятками предметів сітка 7×7 майже напевно щось")
print("загубить. Саме тому в першій YOLO сцени зі зграями дрібних предметів були")
print("найслабшим місцем — і саме тому сітку в наступних версіях зробили густішою.")

## 6 · Навіщо корінь із ширини й висоти

Уяви дві рамки: одна шириною 20 пікселів, друга — 200. Мережа обидві промахнула
рівно на 5 пікселів. Для квадратичної помилки це **однакова** біда: `5² = 25` в обох
випадках. Для ока — зовсім різна: 5 пікселів на рамці 20 px псують рамку, на рамці
200 px їх не видно.

Порахуємо, що при цьому робиться з IoU, і що дає корінь.

In [ ]:
print("ширина   (Δw)²    (Δ√w)²     IoU при похибці 5 px")
for width in (20.0, 50.0, 100.0, 200.0):
    error = 5.0
    plain_penalty = (width - (width + error)) ** 2
    root_penalty = (math.sqrt(width) - math.sqrt(width + error)) ** 2
    # два концентричні квадрати зі сторонами width і width+5
    iou = (width * width) / ((width + error) ** 2)
    print("%6.0f   %6.1f   %7.5f   %.4f" % (width, plain_penalty, root_penalty, iou))

small = (math.sqrt(20) - math.sqrt(25)) ** 2
large = (math.sqrt(200) - math.sqrt(205)) ** 2
print()
print("без кореня штраф на рамці 20 px і на рамці 200 px відноситься як %.2f до 1"
      % ((20 - 25) ** 2 / (200 - 205) ** 2))
print("з коренем — як %.2f до 1" % (small / large))
print()
print("IoU при тій самій похибці: %.4f на малій рамці проти %.4f на великій."
      % (400 / 625, 40000 / 42025))
print("Корінь не вигадує нову метрику — він лише робить штраф схожим на шкоду.")

## 7 · Мережа й функція втрати

Тіло мережі те саме, що в темі 23: три блоки «згортка + батчнорм + ReLU + пулінг»
зменшують 64×64 до 8×8. Уся різниця — **у голові**: тепер вона віддає
`B·5 + C = 2·5 + 3 = 13` чисел на клітинку.

In [ ]:
class TinyYolo(nn.Module):
    def __init__(self, grid_size=GRID, boxes_per_cell=BOXES, class_count=CLASS_COUNT):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.body = nn.Sequential(
            block(1, 16), block(16, 32), block(32, 64),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        # 1×1 згортка: та сама голова застосовується до кожної клітинки окремо
        self.head = nn.Conv2d(64, boxes_per_cell * 5 + class_count, 1)

    def forward(self, x):
        return self.head(self.body(x))


def split_head(raw, grid_size=GRID, boxes_per_cell=BOXES, class_count=CLASS_COUNT):
    """Ріже сирий вихід на координати, впевненість і класи."""
    batch = raw.shape[0]
    moved = raw.permute(0, 2, 3, 1)                       # N, S, S, B·5+C
    box_part = moved[..., :boxes_per_cell * 5].reshape(
        batch, grid_size, grid_size, boxes_per_cell, 5)
    coords = torch.sigmoid(box_part[..., :4])             # усі чотири числа в 0…1
    confidence = torch.sigmoid(box_part[..., 4])
    classes = torch.softmax(moved[..., boxes_per_cell * 5:], dim=-1)
    return coords, confidence, classes


probe_model = TinyYolo()
with torch.no_grad():
    raw_output = probe_model(torch.zeros(1, 1, SIZE, SIZE))
coords, confidence, classes = split_head(raw_output)

print("параметрів у мережі:", sum(p.numel() for p in probe_model.parameters()))
print("сирий вихід:      ", tuple(raw_output.shape), "= N ×", BOXES * 5 + CLASS_COUNT,
      "×", GRID, "×", GRID)
print("координати:       ", tuple(coords.shape))
print("впевненість:      ", tuple(confidence.shape))
print("класи:            ", tuple(classes.shape))
print("чисел на сцену:   ", GRID * GRID * (BOXES * 5 + CLASS_COUNT))

Тепер сама втрата. Вона складається з чотирьох доданків, і кожен рахується **не по
всій сітці**, а по своїй частині:

| доданок | де рахується | вага |
|---|---|---|
| координати `x`, `y`, `√w`, `√h` | тільки у **відповідальної** рамки | `λ_coord` |
| впевненість, коли предмет є | тільки у відповідальної рамки | 1 |
| впевненість, коли предмета немає | у **всіх інших** рамок | `λ_noobj` |
| класи | у клітинок із предметом | 1 |

Відповідальною серед `B` рамок клітинки стає та, чий IoU з істинною рамкою більший.
Ціль для її впевненості — **не одиниця, а сам цей IoU**: YOLO вчить мережу
передбачати, наскільки добре вона поставила рамку.

In [ ]:
def yolo_loss(raw, objectness, target, class_target, lambda_coord, lambda_noobj):
    """Втрата YOLO. Повертає загальне число й чотири складники окремо."""
    coords, confidence, class_probability = split_head(raw)
    batch = raw.shape[0]
    predicted = decode_boxes(coords)                     # N, S, S, B, 4 у пікселях

    with torch.no_grad():
        # істинну рамку теж переводимо в пікселі — тією самою функцією
        truth = decode_boxes(target.unsqueeze(3))        # N, S, S, 1, 4
        truth = truth.expand(-1, -1, -1, BOXES, -1)

        # IoU кожної з B передбачених рамок з істинною рамкою її клітинки
        left = torch.max(predicted[..., 0], truth[..., 0])
        top = torch.max(predicted[..., 1], truth[..., 1])
        right = torch.min(predicted[..., 2], truth[..., 2])
        bottom = torch.min(predicted[..., 3], truth[..., 3])
        intersection = (right - left).clamp(min=0) * (bottom - top).clamp(min=0)
        predicted_area = ((predicted[..., 2] - predicted[..., 0]).clamp(min=0)
                          * (predicted[..., 3] - predicted[..., 1]).clamp(min=0))
        truth_area = (truth[..., 2] - truth[..., 0]) * (truth[..., 3] - truth[..., 1])
        iou = intersection / (predicted_area + truth_area - intersection + 1e-9)

        # відповідальна рамка — з найбільшим IoU, і лише в клітинці з предметом
        responsible = torch.zeros_like(iou).scatter_(
            -1, iou.argmax(dim=-1, keepdim=True), 1.0) * objectness.unsqueeze(-1)
        confidence_target = iou * responsible

    background = 1.0 - responsible

    coordinate_error = ((coords[..., 0] - target[..., 0:1]) ** 2
                        + (coords[..., 1] - target[..., 1:2]) ** 2
                        + (coords[..., 2] - target[..., 2:3]) ** 2
                        + (coords[..., 3] - target[..., 3:4]) ** 2)

    loss_coord = lambda_coord * (responsible * coordinate_error).sum() / batch
    loss_object = (responsible * (confidence - confidence_target) ** 2).sum() / batch
    loss_background = lambda_noobj * (background * confidence ** 2).sum() / batch

    one_hot = torch.zeros_like(class_probability).scatter_(
        -1, class_target.unsqueeze(-1), 1.0)
    loss_class = (objectness.unsqueeze(-1)
                  * (class_probability - one_hot) ** 2).sum() / batch

    total = loss_coord + loss_object + loss_background + loss_class
    return total, (loss_coord.item(), loss_object.item(),
                   loss_background.item(), loss_class.item())


def pack(dataset):
    """Готує тензори всього набору: картинки й три частини цілі."""
    images = np.stack([scene[0] for scene in dataset])[:, None]
    objectness, targets, classes, dropped_total = [], [], [], 0
    for _image, boxes, labels in dataset:
        o, t, c, dropped = encode_targets(boxes, labels)
        objectness.append(o); targets.append(t); classes.append(c)
        dropped_total += dropped
    return (torch.from_numpy(images), torch.from_numpy(np.stack(objectness)),
            torch.from_numpy(np.stack(targets)), torch.from_numpy(np.stack(classes)),
            dropped_total)


train_images, train_obj, train_target, train_cls, train_dropped = pack(train_set)
test_images, test_obj, test_target, test_cls, test_dropped = pack(test_set)

objects_in_train = sum(len(scene[1]) for scene in train_set)
print("предметів у навчальному наборі:  %d" % objects_in_train)
print("із них потрапили в ціль:         %d" % int(train_obj.sum().item()))
print("втрачено через конфлікт клітинок: %d (%.2f %%)"
      % (train_dropped, 100 * train_dropped / objects_in_train))
print()
print("рамок у сітці на одну сцену:      %d" % (GRID * GRID * BOXES))
print("з них відповідальних у середньому: %.2f (%.2f %%)"
      % (train_obj.sum().item() / len(train_set),
         100 * train_obj.sum().item() / len(train_set) / (GRID * GRID * BOXES)))

### Навіщо ваги: подивимось на складники **до** будь-якого навчання

Візьмімо свіжу, ще не навчену мережу й порахуємо втрату на одній партії з двома
наборами ваг: рівними (`λ_coord = 1`, `λ_noobj = 1`) і канонічними з першої статті
про YOLO (`λ_coord = 5`, `λ_noobj = 0.5`).

Головне тут — не сума, а **частки**. Відповідальних рамок дві на сцену зі 128,
а фонових — решта 126. Подивимось, що це робить із балансом доданків.

In [ ]:
torch.manual_seed(0)
fresh_model = TinyYolo()
with torch.no_grad():
    fresh_output = fresh_model(train_images[:32])

background_share = {}
print("настройка              coord      obj      noobj      class     разом")
for name, lambda_coord, lambda_noobj in (("рівні 1 / 1", 1.0, 1.0),
                                          ("канонічні 5 / 0.5", 5.0, 0.5)):
    with torch.no_grad():
        total, parts = yolo_loss(fresh_output, train_obj[:32], train_target[:32],
                                 train_cls[:32], lambda_coord, lambda_noobj)
    shares = [100 * p / total.item() for p in parts]
    background_share[name] = shares[2]
    print("%-18s  %6.3f   %6.3f   %7.3f   %6.3f   %7.3f"
          % (name, parts[0], parts[1], parts[2], parts[3], total.item()))
    print("%-18s  %5.1f %%  %5.1f %%   %5.1f %%   %5.1f %%"
          % ("     частка", shares[0], shares[1], shares[2], shares[3]))

print()
print("З рівними вагами фон зʼїдає майже всю втрату: мережі вигідніше просто")
print("мовчати скрізь, ніж учитися ставити рамку. Канонічні ваги піднімають")
print("координати й притискають фон — і баланс міняється, ще нічого не навчивши.")

## 8 · Навчання: канонічні ваги проти рівних

Тепер перевіримо, чи зміна балансу з попередньої клітинки справді щось дає. Обидві
настройки навчаємо **з трьох зерен** — інакше різницю неможливо відрізнити від шуму.

⏱ Ця клітинка йде близько **двох хвилин**: шість навчань приблизно по 19 секунд.

In [ ]:
def train_model(seed, lambda_coord, lambda_noobj, epochs=15):
    """Навчає мережу з нуля. Зерно керує і початковими вагами, і порядком партій."""
    torch.manual_seed(seed)
    model = TinyYolo()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(train_images))
        for start in range(0, len(train_images), 32):
            batch = order[start:start + 32]
            loss, _parts = yolo_loss(model(train_images[batch]), train_obj[batch],
                                     train_target[batch], train_cls[batch],
                                     lambda_coord, lambda_noobj)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


@torch.no_grad()
def predict(model, images):
    """Розкодовує вихід мережі в списки рамок, оцінок і класів — по сцені на список."""
    model.eval()
    coords, confidence, class_probability = split_head(model(images))
    boxes = decode_boxes(coords)

    results = []
    for n in range(len(images)):
        per_cell = class_probability[n].reshape(GRID * GRID, CLASS_COUNT)
        # клас у YOLO один на клітинку, тому обидві рамки клітинки його поділяють
        labels = per_cell.argmax(-1).unsqueeze(-1).expand(-1, BOXES).reshape(-1)
        best_class = per_cell.max(-1).values.unsqueeze(-1).expand(-1, BOXES).reshape(-1)
        scores = confidence[n].reshape(-1) * best_class
        results.append((boxes[n].reshape(-1, 4), scores, labels))
    return results


print("мережа віддає %d рамок на сцену: %d клітинок × %d рамки на клітинку"
      % (GRID * GRID * BOXES, GRID * GRID, BOXES))
print("оцінка рамки = впевненість × імовірність найкращого класу її клітинки")

In [ ]:
def greedy_match(boxes, scores, truth_boxes, iou_threshold):
    """Жадібне зіставлення за спаданням упевненості — те саме, що в темі 23."""
    order = torch.argsort(scores, descending=True)
    taken = [False] * len(truth_boxes)
    hits = torch.zeros(len(order))

    if len(truth_boxes) and len(order):
        overlaps = box_iou(boxes[order], truth_boxes)
        for position in range(len(order)):
            best_value, best_index = -1.0, -1
            for truth_index in range(len(truth_boxes)):
                if taken[truth_index]:
                    continue
                if overlaps[position, truth_index].item() > best_value:
                    best_value = overlaps[position, truth_index].item()
                    best_index = truth_index
            if best_index >= 0 and best_value >= iou_threshold:
                taken[best_index] = True
                hits[position] = 1.0
    return scores[order], hits


def average_precision(scores, hits, truth_count):
    """AP як площа під огинальною кривої точність-повнота."""
    order = torch.argsort(scores, descending=True)
    ordered_hits = hits[order]
    running_hits = torch.cumsum(ordered_hits, 0)
    running_misses = torch.cumsum(1 - ordered_hits, 0)
    precision = running_hits / (running_hits + running_misses)
    recall = running_hits / truth_count

    envelope = precision.clone()
    for i in range(len(envelope) - 2, -1, -1):
        envelope[i] = max(envelope[i].item(), envelope[i + 1].item())

    area, previous_recall = 0.0, 0.0
    for i in range(len(envelope)):
        area += (recall[i].item() - previous_recall) * envelope[i].item()
        previous_recall = recall[i].item()
    return area


ground_truth = [(torch.from_numpy(scene[1]), torch.from_numpy(scene[2]))
                for scene in test_set]


def mean_average_precision(model, iou_threshold=0.5, nms_threshold=0.5):
    """mAP при одному порозі IoU: середнє AP по трьох класах."""
    predictions = predict(model, test_images)
    values = []
    for class_index in range(CLASS_COUNT):
        score_parts, hit_parts, truth_count = [], [], 0
        for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(predictions,
                                                                        ground_truth):
            chosen = (scores >= 1e-4) & (labels == class_index)
            picked_boxes, picked_scores = boxes[chosen], scores[chosen]
            if len(picked_boxes):
                keep = nms(picked_boxes, picked_scores, nms_threshold)
                picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]
            this_truth = truth_boxes[truth_labels == class_index]
            truth_count += len(this_truth)
            ordered, hits = greedy_match(picked_boxes, picked_scores,
                                         this_truth, iou_threshold)
            score_parts.append(ordered)
            hit_parts.append(hits)
        values.append(average_precision(torch.cat(score_parts), torch.cat(hit_parts),
                                        max(1, truth_count)))
    return float(np.mean(values)), values


def working_point(model, confidence_threshold=0.05, nms_threshold=0.5):
    """Скільки рамок лишається й яка при цьому точність і повнота."""
    predictions = predict(model, test_images)
    kept, hit_count = 0, 0
    for (boxes, scores, _labels), (truth_boxes, _t) in zip(predictions, ground_truth):
        chosen = scores >= confidence_threshold
        picked_boxes, picked_scores = boxes[chosen], scores[chosen]
        if len(picked_boxes):
            keep = nms(picked_boxes, picked_scores, nms_threshold)
            picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]
        kept += len(picked_boxes)
        _ordered, hits = greedy_match(picked_boxes, picked_scores, truth_boxes, 0.5)
        hit_count += int(hits.sum().item())
    return kept, hit_count / max(1, kept), hit_count / true_box_count


settings = (("канонічні 5 / 0.5", 5.0, 0.5), ("рівні 1 / 1", 1.0, 1.0))
scores_by_setting = {}
trained_models = {}
total_started = time.time()

for name, lambda_coord, lambda_noobj in settings:
    values = []
    for seed in (0, 1, 2):
        model, seconds = train_model(seed, lambda_coord, lambda_noobj)
        value, _per_class = mean_average_precision(model)
        kept, precision, recall = working_point(model)
        values.append(value)
        if seed == 0:
            trained_models[name] = model
        print("%-18s зерно %d: %5.1f с   mAP@0.5 = %.4f   рамок %3d   "
              "точність %.3f   повнота %.3f"
              % (name, seed, seconds, value, kept, precision, recall))
    scores_by_setting[name] = values
    print("%-18s середнє %.4f, розкид %.4f"
          % (name, float(np.mean(values)), max(values) - min(values)))
    print()

print("усі шість навчань разом: %.0f с" % (time.time() - total_started))

In [ ]:
canonical = scores_by_setting["канонічні 5 / 0.5"]
equal = scores_by_setting["рівні 1 / 1"]

print("зерно   канонічні 5/0.5   рівні 1/1    різниця")
for seed in range(3):
    print("  %d        %.4f          %.4f      %+.4f"
          % (seed, canonical[seed], equal[seed], canonical[seed] - equal[seed]))
print("  " + "-" * 46)
print("сер.     %.4f          %.4f      %+.4f"
      % (float(np.mean(canonical)), float(np.mean(equal)),
         float(np.mean(canonical)) - float(np.mean(equal))))
print("розкид   %.4f          %.4f"
      % (max(canonical) - min(canonical), max(equal) - min(equal)))

wins = sum(1 for seed in range(3) if canonical[seed] > equal[seed])
gap = float(np.mean(canonical)) - float(np.mean(equal))
biggest_spread = max(max(canonical) - min(canonical), max(equal) - min(equal))

print()
print("Канонічні ваги виграли на %d зернах із 3, у середньому на %.4f." % (wins, gap))
if gap < biggest_spread:
    print("Але розкид усередині однієї настройки більший (%.4f), тому чесний" % biggest_spread)
    print("висновок такий: НА НАШОМУ ДАТАСЕТІ РІЗНИЦІ НЕМАЄ. Три зерна її не ловлять.")
else:
    print("Розкид усередині настройки менший (%.4f), тому різниця справжня." % biggest_spread)
print()
print("Це не значить, що ваги не потрібні. Їхня роль видно на розкладці втрати")
print("з розділу 7: без них фон важить %.1f %% суми." % background_share["рівні 1 / 1"])
print("Наш датасет легкий — три великі предмети на порожньому полотні, і мережа")
print("виїжджає навіть із поганим балансом. На щільній сцені з двома десятками")
print("дрібних предметів запас міцності закінчується, і ваги починають вирішувати.")

Подивимось на передбачення найкращої мережі очима: три перевірні сцени з рамками
після NMS.

In [ ]:
best_model = trained_models["канонічні 5 / 0.5"]
predictions = predict(best_model, test_images[:3])

figure, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for axis, (image, truth_boxes, truth_labels), (boxes, scores, labels) in zip(
        axes, test_set[:3], predictions):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    for box in truth_boxes:
        axis.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                     fill=False, edgecolor="#0f766e", linewidth=1.6))
    chosen = scores >= 0.15
    picked_boxes, picked_scores = boxes[chosen], scores[chosen]
    if len(picked_boxes):
        keep = nms(picked_boxes, picked_scores, 0.5)
        for index in keep.tolist():
            box = picked_boxes[index]
            axis.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0],
                                         box[3] - box[1], fill=False,
                                         edgecolor="#c2185b", linewidth=1.2,
                                         linestyle="--"))
    axis.set_xticks([]); axis.set_yticks([])
plt.tight_layout()
plt.show()
print("бірюзові суцільні — істина, малинові пунктирні — передбачення YOLO після NMS")

## 9 · mAP@0.5 і mAP@0.5:0.95

Ті самі дві метрики, що в [темі 23](../23-detection-setup/lecture.html): перша питає
«рамка приблизно там?», друга усереднює по десяти порогах IoU від 0.50 до 0.95 і питає
«рамка точна?».

In [ ]:
thresholds = [round(float(t), 2) for t in np.arange(0.5, 0.951, 0.05)]
curve = []
for threshold in thresholds:
    value, _per_class = mean_average_precision(best_model, iou_threshold=threshold)
    curve.append(value)

map_50 = curve[0]
map_50_95 = float(np.mean(curve))

print("поріг IoU   mAP")
for threshold, value in zip(thresholds, curve):
    print("   %.2f      %.4f" % (threshold, value))
print()
print("mAP@0.5      = %.4f" % map_50)
print("mAP@0.5:0.95 = %.4f" % map_50_95)
print()
print("Для порівняння, детектор із теми 23 на тих самих сценах давав")
print("mAP@0.5 = 0.9393 і mAP@0.5:0.95 = 0.4900.")

## 10 · Швидкість: один прохід проти багатьох

Одноетапна ідея коштує рівно **один** прохід мережі по зображенню. Двоетапна схема
з [теми 24](../24-two-stage/lecture.html) в її початковому вигляді (R-CNN) проганяє
мережу по **кожній кандидат-області окремо**.

Щоб порівняння було чесним, зробимо його двічі:

1. **при однаковій ємності** — те саме тіло, ті самі ваги, але прогнане по 100
   вирізаних кандидат-областях замість одного цілого зображення;
2. **проти справжнього Faster R-CNN** із `torchvision` — тут різниця включає ще й
   те, що модель у сотні разів більша, і це треба назвати вголос.

In [ ]:
best_model.eval()
single_scene = test_images[:1]

with torch.no_grad():
    for _warmup in range(3):                   # перші проходи завжди повільніші
        best_model(single_scene)
    samples = []
    for _ in range(40):
        started = time.time()
        best_model(single_scene)
        samples.append(time.time() - started)
one_stage_ms = float(np.median(samples)) * 1000
print("одноетапна: один прохід по сцені        %6.2f мс" % one_stage_ms)


class ProposalClassifier(nn.Module):
    """Те саме тіло, але застосоване до вирізаної кандидат-області 32×32."""
    def __init__(self):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.body = nn.Sequential(block(1, 16), block(16, 32), block(32, 64))
        self.head = nn.Linear(64 * 4 * 4, CLASS_COUNT + 1 + 4)

    def forward(self, x):
        return self.head(self.body(x).flatten(1))


proposals = torch.rand(100, 1, 32, 32)
proposal_model = ProposalClassifier()
proposal_model.eval()
with torch.no_grad():
    for _warmup in range(2):
        proposal_model(proposals)
    samples = []
    for _ in range(20):
        started = time.time()
        proposal_model(proposals)
        samples.append(time.time() - started)
two_stage_ms = float(np.median(samples)) * 1000

print("двоетапна: те саме тіло на 100 кандидат-областях %6.1f мс" % two_stage_ms)
print("відношення при однаковій ємності:          %6.0f разів"
      % (two_stage_ms / one_stage_ms))

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# weights=None і weights_backbone=None — архітектура будується офлайн,
# нічого не завантажується
faster = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None,
                                 num_classes=CLASS_COUNT + 1,
                                 min_size=SIZE, max_size=SIZE)
faster.eval()
three_channel = single_scene.repeat(1, 3, 1, 1)[0]

with torch.no_grad():
    faster([three_channel])                    # прогрів
    samples = []
    for _ in range(8):
        started = time.time()
        faster([three_channel])
        samples.append(time.time() - started)
faster_ms = float(np.median(samples)) * 1000

own_parameters = sum(p.numel() for p in best_model.parameters())
faster_parameters = sum(p.numel() for p in faster.parameters())

print("наша YOLO:              %8.2f мс, параметрів %10d" % (one_stage_ms, own_parameters))
print("fasterrcnn_resnet50_fpn %8.0f мс, параметрів %10d" % (faster_ms, faster_parameters))
print()
print("відношення часу       %5.0f разів" % (faster_ms / one_stage_ms))
print("відношення параметрів %5.0f разів" % (faster_parameters / own_parameters))
print()
print("Чесно: більша частина цієї різниці — розмір моделі, а не постановка задачі.")
print("Внесок самої постановки видно в попередній клітинці: %.0f разів при однаковій"
      % (two_stage_ms / one_stage_ms))
print("ємності, лише за те, що тіло проганяється сто разів замість одного.")

## 11 · Що стало з головою через сім версій

Ваг YOLO ми не завантажуємо, але **архітектуру** зібрати офлайн можна: конфігурація
лежить усередині пакета `ultralytics`. Подивимось на голову YOLOv8 і порівняймо її
з першою версією.

Дивись не на розмір, а на **склад** чисел: що в них є й чого в них більше немає.

In [ ]:
try:
    from ultralytics.nn.tasks import DetectionModel

    # ваги не завантажуються: модель будується з yaml-конфігурації пакета
    modern = DetectionModel("yolov8n.yaml", nc=80, verbose=False)
    head = modern.model[-1]
    positions = 80 * 80 + 40 * 40 + 20 * 20        # три масштаби при вході 640×640

    print("тип голови              ", type(head).__name__)
    print("масштабів (рівнів)       %d" % head.nl)
    print("чисел на позицію         %d = %d класів + 4 × %d"
          % (head.no, head.nc, head.reg_max))
    print("окремі гілки голови      рамка %d, класи %d" % (len(head.cv2), len(head.cv3)))
    print("позицій сітки при 640    %d" % positions)
    print("чисел разом              %d" % (positions * head.no))
    print("параметрів у моделі      %d" % sum(p.numel() for p in modern.parameters()))
    print()
    print("Для порівняння, YOLOv1: 1 масштаб, 49 позицій, 30 чисел на позицію,")
    print("разом 1470.")
    print()
    print("Найважливіше в цих числах — чого в них НЕМАЄ. Окремого числа")
    print("«тут є предмет» у YOLOv8 не лишилось: його роль перебрала найбільша")
    print("ймовірність класу. А 4 × %d означає, що кожна сторона рамки" % head.reg_max)
    print("передбачається розподілом по %d позиціях, а не одним числом." % head.reg_max)
    print("Дві окремі гілки — це та сама розділена голова з лекції.")
except Exception as error:
    # пакет може бути не встановлений — тоді просто друкуємо числа з лекції
    print("ultralytics недоступний:", type(error).__name__, error)
    print("Числа з лекції: 3 масштаби, 8400 позицій, 144 числа на позицію,")
    print("разом 1 209 600 проти 1 470 у YOLOv1.")

## 12 · Підсумок зошита

In [ ]:
sparse_total, sparse_lost, _d = count_conflicts(sparse_set, 8)
dense_total, dense_lost, dense_damaged = count_conflicts(dense_set, 8)
dense_total4, dense_lost4, dense_damaged4 = count_conflicts(dense_set, 4)

print("Що поміряно:")
print()
print("  розмір виходу 8×8×13                  %d чисел на сцену"
      % (GRID * GRID * (BOXES * 5 + CLASS_COUNT)))
print("  розмір виходу YOLOv1 (7×7×30)         %d чисел" % (7 * 7 * 30))
print("  втрачено сіткою 8×8, рідкі сцени      %d із %d (%.2f %%)"
      % (sparse_lost, sparse_total, 100 * sparse_lost / sparse_total))
print("  втрачено сіткою 8×8, щільні сцени     %d із %d (%.2f %%)"
      % (dense_lost, dense_total, 100 * dense_lost / dense_total))
print("  втрачено сіткою 4×4, щільні сцени     %d із %d (%.2f %%), зіпсовано %.1f %% сцен"
      % (dense_lost4, dense_total4, 100 * dense_lost4 / dense_total4,
         100 * dense_damaged4 / len(dense_set)))
print("  штраф за 5 px: без кореня             однаковий на рамках 20 і 200 px")
print("  штраф за 5 px: з коренем              у %.1f раза більший на малій рамці"
      % ((math.sqrt(20) - math.sqrt(25)) ** 2 / (math.sqrt(200) - math.sqrt(205)) ** 2))
print("  частка фону у втраті з рівними вагами %.1f %%" % background_share["рівні 1 / 1"])
print("  те саме з канонічними вагами          %.1f %%"
      % background_share["канонічні 5 / 0.5"])
print("  mAP@0.5, канонічні ваги               %.4f (середнє з трьох зерен)"
      % float(np.mean(canonical)))
print("  mAP@0.5, рівні ваги                   %.4f (середнє з трьох зерен)"
      % float(np.mean(equal)))
print("  mAP@0.5 найкращої мережі              %.4f" % map_50)
print("  mAP@0.5:0.95 найкращої мережі         %.4f" % map_50_95)
print("  один прохід одноетапної               %.2f мс" % one_stage_ms)
print("  сто кандидат-областей тим самим тілом        %.1f мс (%.0f разів)"
      % (two_stage_ms, two_stage_ms / one_stage_ms))
print()
print("Головне число теми — втрата на щільних сценах. Сітка 4×4 не бачить кожен")
print("одинадцятий предмет у принципі, і жодне навчання цього не виправляє.")

## Завдання

### 🟢 Рівень 1 — База

Прожени `count_conflicts` для `S` від 2 до 16 на `dense_set` і побудуй графік «частка
втрачених предметів проти `S`». Познач на ньому точку `S = 7` — сітку першої YOLO.

**Зроблено, якщо:** графік побудовано, і ти можеш назвати найменше `S`, при якому
втрата на щільних сценах падає нижче за 1 %.

### 🟡 Рівень 2 — Плюс

Навчи мережу з `BOXES = 1` (одна рамка на клітинку) і порівняй `mAP@0.5` із нашою
двохрамковою. Заразом подивись, чи справді дві рамки спеціалізуються: для навченої
мережі порахуй середнє відношення ширини до висоти окремо для рамки 0 і рамки 1.

**Зроблено, якщо:** два числа mAP надруковано, і ти можеш сказати, чи розійшлись
форми двох рамок, спираючись на числа, а не на враження.

### 🔴 Рівень 3 — Виклик

Збери повний декодер: вихід мережі → рамки → поріг упевненості → NMS → список
`(рамка, клас, оцінка)` — і прожени ним обидва детектори: наш YOLO-подібний і той,
що навчався в темі 23. Порівняй `mAP@0.5` і `mAP@0.5:0.95`.

**Зроблено, якщо:** чотири числа надруковано в одній таблиці, і ти можеш пояснити
різницю, спираючись на різницю в правилі призначення відповідальності: у темі 23
позитивною була **кожна** клітинка всередині рамки, у YOLO — **одна**.